# Cornish-Fisher Expansion



$$
Z^{C.-F.}_{_{\alpha}} \approx Z_{\alpha} + \frac{1}{6}(Z_{\alpha}^2 - 1)S + \frac{1}{24}(Z_{\alpha}^3 - 3Z_{\alpha})K - \frac{1}{36}(2Z_{\alpha}^3 - 5Z_{\alpha})S^2
$$


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm, skew, kurtosis

def cornish_fisher_var(pnl_series, alpha=0.01):
    mean = np.mean(pnl_series)
    std = np.std(pnl_series, ddof=0)
    s = skew(pnl_series)
    k = kurtosis(pnl_series)
    z = norm.ppf(alpha)
    z_cf = (z +
            (z**2 - 1) * s / 6 +
            (z**3 - 3*z) * k / 24 -
            (2 * z**3 - 5 * z) * s**2 / 36)
    var_cf = -(mean + z_cf * std)
    return var_cf

def historical_var(pnl_series, alpha=0.01):
    return -np.percentile(pnl_series, alpha*100)  # negative for loss

# Read your file
df = pd.read_csv('CF.csv')

# Calculate and format results in millions with no decimal places
result = df.groupby('date')['PnL'].agg(
    VaR=lambda x: round(historical_var(x, alpha=0.01)/1000000),
    CFVaR99=lambda x: round(cornish_fisher_var(x, alpha=0.01)/1000000),
    CFVaR9999=lambda x: round(cornish_fisher_var(x, alpha=0.0001)/1000000)
    
).reset_index()

# Convert date column to datetime (if needed)
result['date'] = pd.to_datetime(result['date'])

# Create a flag for 2025-07-31
flag = result['date'] == pd.Timestamp('2025-07-31')

# Concatenate: 2025-07-31 row at top, then rest sorted descending
result = pd.concat([result[flag], result[~flag].sort_values('date', ascending=False)], ignore_index=True)

print(result)

         date  VaR  CFVaR99  CFVaR9999
0  2025-07-31  189      179        298
1  2025-06-30  187      176        282
2  2025-05-31  177      168        256
3  2025-04-30  164      167        248
4  2025-03-31  187      178        266
5  2025-02-28  178      172        262
6  2025-01-31  198      191        321
7  2024-12-31  196      204        335
8  2024-11-30  180      185        307
9  2024-10-31  176      186        303
10 2024-09-30  173      178        275
11 2024-08-31  160      164        245
12 2024-07-31  180      179        307
13 2024-06-30  165      160        278
14 2024-05-31  155      146        244
15 2024-04-30  164      171        319
16 2024-03-31  162      173        329
17 2024-02-29  151      160        281
18 2024-01-31  147      160        273
19 2023-12-31  145      153        259
20 2023-11-30  141      148        258
21 2023-10-31  134      139        280
22 2023-09-30  147      152        310
23 2023-08-31  147      148        301
24 2023-07-31  138      1

In [52]:
import pandas as pd
import numpy as np
from scipy.stats import norm, skew, kurtosis

def cornish_fisher_var(pnl_series, alpha=0.01):
    mean = np.mean(pnl_series)
    std = np.std(pnl_series, ddof=0)
    s = skew(pnl_series)
    k = kurtosis(pnl_series)
    z = norm.ppf(alpha)
    z_cf = (z +
            (z**2 - 1) * s / 6 +
            (z**3 - 3*z) * k / 24 -
            (2 * z**3 - 5 * z) * s**2 / 36)
    var_cf = -(mean + z_cf * std)
    return var_cf

def historical_var(pnl_series, alpha=0.01):
    return -np.percentile(pnl_series, alpha*100)

def scale_var_9999_normal(var99):
    z_99 = abs(norm.ppf(0.01))
    z_9999 = abs(norm.ppf(0.0001))
    scale_factor = z_9999 / z_99
    return var99 * scale_factor

# Read your file
df = pd.read_excel('Month-End_PnL_Distributions.xlsx')

def group_aggregator(x):
    mean = np.mean(x) / 1_000_000
    std = np.std(x, ddof=0) / 1_000_000
    s = skew(x)
    k = kurtosis(x)
    var90 = historical_var(x, alpha=0.10)
    var95 = historical_var(x, alpha=0.05)
    var9772 = historical_var(x, alpha=0.0228)
    var99 = historical_var(x, alpha=0.01)
    cfvar99 = cornish_fisher_var(x, alpha=0.01)
    var9999_normal = scale_var_9999_normal(var99)
    cfvar9999 = cornish_fisher_var(x, alpha=0.0001)
    return pd.Series({
        'mean': mean,
        'std': std,
        's': s,
        'k': k,
        'VaR90': round(var90 / 1_000_000),
        'VaR95': round(var95 / 1_000_000),
        'VaR9772': round(var9772 / 1_000_000),
        'VaR99': round(var99 / 1_000_000),
        'CFVAR99': round(cfvar99 / 1_000_000),
        'VaR9999_normal_scaled': round(var9999_normal / 1_000_000),
        'CFVAR9999': round(cfvar9999 / 1_000_000)
    })

# Perform groupby and get a DataFrame with proper columns
result = df.groupby('Month-End')['PnL Distribution'].apply(group_aggregator).unstack().reset_index()

# Convert 'Month-End' column to datetime (if not already)
result['Month-End'] = pd.to_datetime(result['Month-End'])

# Place '2025-07-31' at the top, others sorted descending
top_row = result['Month-End'] == pd.Timestamp('2025-07-31')
result = pd.concat([
    result[top_row],
    result[~top_row].sort_values('Month-End', ascending=False)
], ignore_index=True)

# Reorder columns according to your requirements
result = result[['Month-End', 'mean', 'std', 's', 'k', 'VaR90', 'VaR95', 'VaR99', 'VaR9772', 'CFVAR99', 'VaR9999_normal_scaled', 'CFVAR9999']]

print(result)
result.to_excel('VaR_results.xlsx', index=False)


    Month-End       mean        std         s         k  VaR90  VaR95  VaR99  \
0  2025-07-31  12.479280  81.327030 -0.024500  0.026406   82.0  133.0  189.0   
1  2025-06-30  10.968214  80.740270  0.007693 -0.042732   86.0  129.0  187.0   
2  2025-05-31   8.483642  77.715267  0.027618 -0.148606   86.0  124.0  177.0   
3  2025-04-30   7.678327  78.079140  0.060922 -0.178262   88.0  125.0  164.0   
4  2025-03-31   4.361401  80.728360  0.030005 -0.178454   92.0  130.0  187.0   
5  2025-02-28   2.091477  77.184102  0.055109 -0.105039   90.0  128.0  178.0   
6  2025-01-31   4.458452  97.907095  0.589512  1.001990  109.0  145.0  198.0   
7  2024-12-31  -2.099327  99.958216  0.544497  0.873121  120.0  156.0  196.0   
8  2024-11-30   2.462067  92.918110  0.539590  0.863519  105.0  139.0  180.0   
9  2024-10-31   4.268807  92.030311  0.438553  0.599613  108.0  138.0  176.0   
10 2024-09-30  12.276061  93.932605  0.446356  0.452359   99.0  130.0  173.0   
11 2024-08-31   7.918869  86.297051  0.4

In [92]:
import pandas as pd
import numpy as np
from scipy.stats import norm, skew, kurtosis

def parametric_normal_var(mean, std, alpha):
    z = norm.ppf(alpha)
    return -(mean + z * std)

def parametric_normal_es(mean, std, alpha):
    z = norm.ppf(alpha)
    return (mean + std * norm.pdf(z) / (alpha))

def cornish_fisher_var(pnl_series, alpha=0.01):
    mean = np.mean(pnl_series)
    std = np.std(pnl_series, ddof=0)
    s = skew(pnl_series)
    k = kurtosis(pnl_series)
    z = norm.ppf(alpha)
    z_cf = (z +
            (z**2 - 1) * s / 6 +
            (z**3 - 3*z) * k / 24 -
            (2 * z**3 - 5 * z) * s**2 / 36)
    var_cf = -(mean + z_cf * std)
    return var_cf

def historical_var(pnl_series, alpha=0.01):
    return -np.percentile(pnl_series, alpha * 100)

def scale_var_9999_normal(var99):
    z_99 = abs(norm.ppf(0.01))
    z_9999 = abs(norm.ppf(0.0001))
    scale_factor = z_9999 / z_99
    return var99 * scale_factor

def historical_es(pnl_series, alpha=0.01):
    var = np.percentile(pnl_series, alpha * 100)
    return -np.mean(pnl_series[pnl_series <= var])

# Read your file
df = pd.read_excel('Month_End_PnL_Distributions.xlsx')


def group_aggregator(x):
    mean = np.mean(x)
    std = np.std(x, ddof=0)
    s = skew(x)
    k = kurtosis(x)
    # Parametric normal VaR and ES at key levels
    normvar90 = parametric_normal_var(mean, std, 0.10)
    normvar95 = parametric_normal_var(mean, std, 0.05)
    normvar9772 = parametric_normal_var(mean, std, 0.0228)
    normvar99 = parametric_normal_var(mean, std, 0.01)
    normes90 = parametric_normal_es(mean, std, 0.10)
    normes95 = parametric_normal_es(mean, std, 0.05)
    normes9772 = parametric_normal_es(mean, std, 0.0228)
    normes99 = parametric_normal_es(mean, std, 0.01)
    # Empirical VaR/ES
    var90 = historical_var(x, alpha=0.10)
    var95 = historical_var(x, alpha=0.05)
    var9772 = historical_var(x, alpha=0.0228)
    var99 = historical_var(x, alpha=0.01)
    es90 = historical_es(x, alpha=0.10)
    es95 = historical_es(x, alpha=0.05)
    es9772 = historical_es(x, alpha=0.0228)
    es99 = historical_es(x, alpha=0.01)
    # Cornish-Fisher/extra metrics
    cfvar99 = cornish_fisher_var(x, alpha=0.01)
    var9999_normal = scale_var_9999_normal(var99)
    cfvar9999 = cornish_fisher_var(x, alpha=0.0001)
    return pd.Series({
        'Mean': round(mean / 1_000_000, 1),
        'Std': round(std / 1_000_000),
        'Skew': round(s, 2),
        'Excess K': round(k, 2),
        'NormVaR90': round(normvar90 / 1_000_000),
        'NormES90': round(normes90 / 1_000_000),
        'NormVaR95': round(normvar95 / 1_000_000),
        'NormES95': round(normes95 / 1_000_000),
        'NormVaR97.7': round(normvar9772 / 1_000_000),
        'NormES97.7': round(normes9772 / 1_000_000),
        'NormVaR99': round(normvar99 / 1_000_000),
        'NormES99': round(normes99 / 1_000_000),
        'VaR90': round(var90 / 1_000_000),
        'ES90': round(es90 / 1_000_000),
        'VaR95': round(var95 / 1_000_000),
        'ES95': round(es95 / 1_000_000),
        'VaR97.7': round(var9772 / 1_000_000),
        'ES97.7': round(es9772 / 1_000_000),
        'VaR99': round(var99 / 1_000_000),
        'ES99': round(es99 / 1_000_000),
        'CFVAR99': round(cfvar99 / 1_000_000),
        'VaR99.99_normal_scaled': round(var9999_normal / 1_000_000),
        'CFVAR99.99': round(cfvar9999 / 1_000_000)
    })

result = df.groupby('Month_End')['PnL_Distribution'].apply(group_aggregator).unstack().reset_index()

result['Month_End'] = pd.to_datetime(result['Month_End'])
top_row = result['Month_End'] == pd.Timestamp('2025-07-31')
result = pd.concat([
    result[top_row],
    result[~top_row].sort_values('Month_End', ascending=False)
], ignore_index=True)

result = result[['Month_End', 'Mean', 'Std', 'Skew', 'Excess K',
    'VaR90', 'ES90', 'NormVaR90', 'NormES90', 
    'VaR95', 'ES95', 'NormVaR95', 'NormES95', 
    'VaR97.7', 'ES97.7', 'NormVaR97.7', 'NormES97.7', 
    'VaR99', 'ES99', 'NormVaR99', 'NormES99', 
    'CFVAR99', 'VaR99.99_normal_scaled', 'CFVAR99.99']]

print(result)
result.to_excel('VaR_Metrics.xlsx', index=False)


    Month_End  Mean    Std  Skew  Excess K  VaR90   ES90  NormVaR90  NormES90  \
0  2025-07-31  12.5   81.0 -0.02      0.03   82.0  133.0       92.0     155.0   
1  2025-06-30  11.0   81.0  0.01     -0.04   86.0  132.0       93.0     153.0   
2  2025-05-31   8.5   78.0  0.03     -0.15   86.0  129.0       91.0     145.0   
3  2025-04-30   7.7   78.0  0.06     -0.18   88.0  128.0       92.0     145.0   
4  2025-03-31   4.4   81.0  0.03     -0.18   92.0  137.0       99.0     146.0   
5  2025-02-28   2.1   77.0  0.06     -0.11   90.0  134.0       97.0     138.0   
6  2025-01-31   4.5   98.0  0.59      1.00  109.0  152.0      121.0     176.0   
7  2024-12-31  -2.1  100.0  0.54      0.87  120.0  162.0      130.0     173.0   
8  2024-11-30   2.5   93.0  0.54      0.86  105.0  146.0      117.0     166.0   
9  2024-10-31   4.3   92.0  0.44      0.60  108.0  146.0      114.0     166.0   
10 2024-09-30  12.3   94.0  0.45      0.45   99.0  138.0      108.0     177.0   
11 2024-08-31   7.9   86.0  